# Retail Sales Analytics & Profit Prediction\nTeaching project using the supplied synthetic dataset. Replace it with authorized real data if required.

In [ ]:
import pandas as pd, numpy as np\nimport matplotlib.pyplot as plt, seaborn as sns\ndf = pd.read_csv('../data/dataset.csv', parse_dates=['Order_Date'])\ndf.shape, df.head()

In [ ]:
df.info()\ndf.describe(include='all').T\ndf.isna().sum(), df.duplicated().sum()

In [ ]:
df = df.drop_duplicates().copy()\ndf['Order_Year'] = df.Order_Date.dt.year\ndf['Order_Month'] = df.Order_Date.dt.month\ndf['Profit_Margin'] = np.where(df.Total_Sale > 0, df.Profit / df.Total_Sale * 100, 0)\ndf['Discount_Amount'] = df.Quantity * df.Unit_Price * df.Discount\ndf['Delivery_Status'] = np.where(df.Delivery_Days > 7, 'Late', 'On Time')\ndf.head()

In [ ]:
print('Revenue:', df.Total_Sale.sum())\nprint('Profit:', df.Profit.sum())\nprint('Average order value:', df.Total_Sale.mean())\nprint('Return rate:', (df.Returned == 'Yes').mean())\ndf.groupby('Product_Category')[['Total_Sale','Profit']].sum().sort_values('Profit', ascending=False)

In [ ]:
sns.lineplot(data=df.groupby(df.Order_Date.dt.to_period('M')).Total_Sale.sum().reset_index(), x='Order_Date', y='Total_Sale')\nplt.xticks(rotation=45); plt.title('Monthly sales'); plt.tight_layout()

In [ ]:
from sklearn.model_selection import train_test_split\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.tree import DecisionTreeRegressor\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfeatures=['Customer_Gender','Customer_Age','City','Region','Product_Category','Product_Name','Brand','Quantity','Unit_Price','Discount','Shipping_Cost','Payment_Method','Delivery_Days','Salesperson','Order_Year','Order_Month','Quarter']\nX=df[features]; y=df.Profit\ncat_cols=X.select_dtypes(include='object').columns; num_cols=X.select_dtypes(exclude='object').columns\nprep=ColumnTransformer([('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat_cols),('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols)])\nX_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)\nfor name,est in [('Linear Regression',LinearRegression()),('Decision Tree',DecisionTreeRegressor(max_depth=12,random_state=42)),('Random Forest',RandomForestRegressor(n_estimators=180,random_state=42,n_jobs=-1))]:\n    pipe=Pipeline([('prep',prep),('model',est)]); pipe.fit(X_train,y_train); p=pipe.predict(X_test)\n    print(name, 'MAE',mean_absolute_error(y_test,p),'MSE',mean_squared_error(y_test,p),'RMSE',mean_squared_error(y_test,p)**.5,'R2',r2_score(y_test,p))

## Leakage prevention\nOrder_ID, Customer_ID and Customer_Name are identifiers/personal names and are excluded. Customer_Rating and Returned are outcomes not reliably available before order completion. Total_Sale is also excluded because it is a post-calculation value and can make a prediction unrealistic. Profit_Margin and other target-derived features are not model inputs.

## Interpretation\nTreat patterns as synthetic-data demonstrations. Correlation does not establish causation. Verify findings using real business records before making operational decisions.